In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os
import warnings

warnings.filterwarnings("ignore")

## <다항 회귀>

In [2]:
print("""
응답변수와 예측변수간의 관계가 반드시 선형일 필요는 없다. 비선형 효과를 회귀 분석에 담기위해
회귀모형을 확장하는 여러가지 방법들이 있다.

1. 다항 회귀
2. 스플라인 회귀

""")


응답변수와 예측변수간의 관계가 반드시 선형일 필요는 없다. 비선형 효과를 회귀 분석에 담기위해
회귀모형을 확장하는 여러가지 방법들이 있다.

1. 다항 회귀
2. 스플라인 회귀




In [3]:
print("""
다항회귀는 2차 함수 이상의 다항 함수를 이용하여 두 변수간의 관계를 설명하는 알고리즘이다.
단순 선형 모델의 한계를 일부 보완할 수 있다.
""")


다항회귀는 2차 함수 이상의 다항 함수를 이용하여 두 변수간의 관계를 설명하는 알고리즘이다.
단순 선형 모델의 한계를 일부 보완할 수 있다.



In [29]:
from sklearn.datasets import load_diabetes

data = load_diabetes()

X = pd.DataFrame(data["data"],
                 columns=data["feature_names"]
                 )
y = pd.Series(data["target"], name="target")

y

0      151.0
1       75.0
2      141.0
3      206.0
4      135.0
       ...  
437    178.0
438    104.0
439    132.0
440    220.0
441     57.0
Name: target, Length: 442, dtype: float64

In [30]:
X = X[["age", "sex", "bmi", "bp"]]

In [31]:
data = pd.concat([X, y], axis=1)
data

,age,sex,bmi,bp,target
0,0.038076,0.050680,0.061696,0.021872,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,75.0
2,0.085299,0.050680,0.044451,-0.005671,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,206.0
4,0.005383,-0.044642,-0.036385,0.021872,135.0
...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,104.0
439,0.041708,0.050680,-0.015906,0.017282,132.0
440,-0.045472,-0.044642,0.039062,0.001215,220.0


In [32]:
## 원본 데이터
data

,age,sex,bmi,bp,target
0,0.038076,0.050680,0.061696,0.021872,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,75.0
2,0.085299,0.050680,0.044451,-0.005671,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,206.0
4,0.005383,-0.044642,-0.036385,0.021872,135.0
...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,104.0
439,0.041708,0.050680,-0.015906,0.017282,132.0
440,-0.045472,-0.044642,0.039062,0.001215,220.0


In [33]:
### 독립변수 종속변수 구분
X = data[["age", "sex", "bmi", "bp"]]
y = data["target"]

In [34]:
### 독립변수 다차원 변환
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
poly_array = poly.fit_transform(X)

X_poly = pd.DataFrame(poly_array)
X_poly

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,1.0,0.038076,0.050680,0.061696,0.021872,0.001450,0.001930,0.002349,0.000833,0.002568,0.003127,0.001108,0.003806,0.001349,0.000478
1,1.0,-0.001882,-0.044642,-0.051474,-0.026328,0.000004,0.000084,0.000097,0.000050,0.001993,0.002298,0.001175,0.002650,0.001355,0.000693
2,1.0,0.085299,0.050680,0.044451,-0.005671,0.007276,0.004323,0.003792,-0.000484,0.002568,0.002253,-0.000287,0.001976,-0.000252,0.000032
3,1.0,-0.089063,-0.044642,-0.011595,-0.036656,0.007932,0.003976,0.001033,0.003265,0.001993,0.000518,0.001636,0.000134,0.000425,0.001344
4,1.0,0.005383,-0.044642,-0.036385,0.021872,0.000029,-0.000240,-0.000196,0.000118,0.001993,0.001624,-0.000976,0.001324,-0.000796,0.000478
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
437,1.0,0.041708,0.050680,0.019662,0.059744,0.001740,0.002114,0.000820,0.002492,0.002568,0.000996,0.003028,0.000387,0.001175,0.003569
438,1.0,-0.005515,0.050680,-0.015906,-0.067642,0.000030,-0.000279,0.000088,0.000373,0.002568,-0.000806,-0.003428,0.000253,0.001076,0.004575
439,1.0,0.041708,0.050680,-0.015906,0.017282,0.001740,0.002114,-0.000663,0.000721,0.002568,-0.000806,0.000876,0.000253,-0.000275,0.000299
440,1.0,-0.045472,-0.044642,0.039062,0.001215,0.002068,0.002030,-0.001776,-0.000055,0.001993,-0.001744,-0.000054,0.001526,0.000047,0.000001


In [37]:
### 데이터 분할
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(X_poly, y,
                                                                      test_size=0.2,
                                                                      random_state=42
                                                                      )
print(train_input.shape, test_input.shape, train_target.shape, test_target.shape)

(353, 15) (89, 15) (353,) (89,)


In [42]:
### 다중 선형회귀 모델 적합
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

lr = LinearRegression()
lr.fit(train_input, train_target)

# 예측
lr_pred = lr.predict(test_input)

# 평가
r2 = r2_score(test_target, lr_pred)
r2

0.3455030093132728

In [44]:
## statsmodels 방식

In [45]:
## 원본 데이터
data

,age,sex,bmi,bp,target
0,0.038076,0.050680,0.061696,0.021872,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,75.0
2,0.085299,0.050680,0.044451,-0.005671,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,206.0
4,0.005383,-0.044642,-0.036385,0.021872,135.0
...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,104.0
439,0.041708,0.050680,-0.015906,0.017282,132.0
440,-0.045472,-0.044642,0.039062,0.001215,220.0


In [46]:
### 독립변수 종속변수 구분
X = data[["age", "sex", "bmi", "bp"]]
y = data["target"]

In [47]:
### 독립변수 다차원 변환
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
poly_array = poly.fit_transform(X)

X_poly = pd.DataFrame(poly_array)
X_poly

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,1.0,0.038076,0.050680,0.061696,0.021872,0.001450,0.001930,0.002349,0.000833,0.002568,0.003127,0.001108,0.003806,0.001349,0.000478
1,1.0,-0.001882,-0.044642,-0.051474,-0.026328,0.000004,0.000084,0.000097,0.000050,0.001993,0.002298,0.001175,0.002650,0.001355,0.000693
2,1.0,0.085299,0.050680,0.044451,-0.005671,0.007276,0.004323,0.003792,-0.000484,0.002568,0.002253,-0.000287,0.001976,-0.000252,0.000032
3,1.0,-0.089063,-0.044642,-0.011595,-0.036656,0.007932,0.003976,0.001033,0.003265,0.001993,0.000518,0.001636,0.000134,0.000425,0.001344
4,1.0,0.005383,-0.044642,-0.036385,0.021872,0.000029,-0.000240,-0.000196,0.000118,0.001993,0.001624,-0.000976,0.001324,-0.000796,0.000478
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
437,1.0,0.041708,0.050680,0.019662,0.059744,0.001740,0.002114,0.000820,0.002492,0.002568,0.000996,0.003028,0.000387,0.001175,0.003569
438,1.0,-0.005515,0.050680,-0.015906,-0.067642,0.000030,-0.000279,0.000088,0.000373,0.002568,-0.000806,-0.003428,0.000253,0.001076,0.004575
439,1.0,0.041708,0.050680,-0.015906,0.017282,0.001740,0.002114,-0.000663,0.000721,0.002568,-0.000806,0.000876,0.000253,-0.000275,0.000299
440,1.0,-0.045472,-0.044642,0.039062,0.001215,0.002068,0.002030,-0.001776,-0.000055,0.001993,-0.001744,-0.000054,0.001526,0.000047,0.000001


In [48]:
### 데이터 분할
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(X_poly, y,
                                                                      test_size=0.2,
                                                                      random_state=42
                                                                      )
print(train_input.shape, test_input.shape, train_target.shape, test_target.shape)

(353, 15) (89, 15) (353,) (89,)


In [52]:
## 다중 선형회귀 모델 적합
from statsmodels.api import OLS

model = OLS(train_target, train_input).fit()

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 target   R-squared:                       0.447
Model:                            OLS   Adj. R-squared:                  0.425
Method:                 Least Squares   F-statistic:                     21.04
Date:                Sat, 23 Aug 2025   Prob (F-statistic):           2.45e-36
Time:                        23:45:15   Log-Likelihood:                -1934.2
No. Observations:                 353   AIC:                             3896.
Df Residuals:                     339   BIC:                             3950.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
0            145.6303      5.535     26.309      0.000     134.742     156.518
1            111.4584     76.281      1.461      0.145     -38.586     261.502
2           -113.3686     69.458     -1.632      0.104    -249.992      23.254
3            827.0805     87.734      9.427      0.000     654.509     999.652
4            389.4412     79.762      4.883      0.000     232.551     546.331
5            485.6019   1463.106      0.332      0.740   -2392.308    3363.512
6           2378.0875   1570.595      1.514      0.131    -711.251    5467.426
7            495.5396   1784.042      0.278      0.781   -3013.647    4004.727
8           2694.9189   1893.367      1.423      0.156   -1029.308    6419.146
9             -0.3551      0.419     -0.848      0.397      -1.179       0.469
10           931.3009   1642.436      0.567      0.571   -2299.349    4161.951
11          1351.2617   1630.395      0.829      0.408   -1855.704    4558.227
12           636.0378   1272.275      0.500      0.617   -1866.510    3138.586
13          1744.5282   1757.639      0.993      0.322   -1712.723    5201.780
14          -679.6781   1353.395     -0.502      0.616   -3341.787    1982.431
==============================================================================
Omnibus:                        6.501   Durbin-Watson:                   1.782
Prob(Omnibus):                  0.039   Jarque-Bera (JB):                4.832
Skew:                           0.166   Prob(JB):                       0.0893
Kurtosis:                       2.533   Cond. No.                     3.62e+17
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 2.7e-33. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [54]:
### 예측
pred = model.predict(test_input)
pred

287    139.717914
211    200.628821
72     144.998470
321    250.284443
73     124.332319
          ...    
255    100.665665
90     117.111843
57      95.527543
391     87.911984
24     174.756591
Length: 89, dtype: float64

In [56]:
### 평가
from sklearn.metrics import r2_score

r2 = r2_score(test_target, pred)
r2

0.34550300931327305

## <스플라인 회귀>

In [57]:
print("""
스플라인 회귀는 다항 구간들을 부드러운 곡선 형태로 적합하는 방법이다.
스플라인 구간을 구분하는 값을 매듭이라고 한다. 구간별 다항식은 예측변수를 위한 매듭사이를 부드럽게 연결한다.
sklearn의 SplineTransformer()에서는 매듭의 개수와 차수를 지정하므로써 기본 스플라인 항을 적용하게 된다.
""")


스플라인 회귀는 다항 구간들을 부드러운 곡선 형태로 적합하는 방법이다.
스플라인 구간을 구분하는 값을 매듭이라고 한다. 구간별 다항식은 예측변수를 위한 매듭사이를 부드럽게 연결한다.
sklearn의 SplineTransformer()에서는 매듭의 개수와 차수를 지정하므로써 기본 스플라인 항을 적용하게 된다.



In [58]:
from sklearn.datasets import load_diabetes

data = load_diabetes()

X = pd.DataFrame(data["data"],
                 columns=data["feature_names"]
                 )
y = pd.Series(data["target"], name="target")

y

0      151.0
1       75.0
2      141.0
3      206.0
4      135.0
       ...  
437    178.0
438    104.0
439    132.0
440    220.0
441     57.0
Name: target, Length: 442, dtype: float64

In [59]:
X = X[["age", "sex", "bmi", "bp"]]

In [60]:
data = pd.concat([X, y], axis=1)
data

,age,sex,bmi,bp,target
0,0.038076,0.050680,0.061696,0.021872,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,75.0
2,0.085299,0.050680,0.044451,-0.005671,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,206.0
4,0.005383,-0.044642,-0.036385,0.021872,135.0
...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,104.0
439,0.041708,0.050680,-0.015906,0.017282,132.0
440,-0.045472,-0.044642,0.039062,0.001215,220.0


In [62]:
### 원본데이터
data

,age,sex,bmi,bp,target
0,0.038076,0.050680,0.061696,0.021872,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,75.0
2,0.085299,0.050680,0.044451,-0.005671,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,206.0
4,0.005383,-0.044642,-0.036385,0.021872,135.0
...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,104.0
439,0.041708,0.050680,-0.015906,0.017282,132.0
440,-0.045472,-0.044642,0.039062,0.001215,220.0


In [63]:
from sklearn.preprocessing import SplineTransformer

ImportError: cannot import name 'SplineTransformer' from 'sklearn.preprocessing' (C:\Users\jacob\anaconda3\envs\adp_env\lib\site-packages\sklearn\preprocessing\__init__.py)